# Step 2 – From detectors to measurement locations

In step 1 we got data for **32 detectors**. But we want to analyse **12 locations**
(6 eastbound, 6 westbound). Some locations have several detectors:

- **MONICA** detectors measure **one lane** each.
- **MONIBAS** detectors measure **all lanes** of the carriageway together.

If a location has both, adding them up would count the same cars twice.
In this notebook we check which detectors measure the same traffic, choose one source per
location, and make one table with one row per location, date and hour.

In [ ]:
import pandas as pd

data = pd.read_csv("../processed_data/A1_detectors_hourly.csv")

# Shorter detector names are easier to read
data["short"] = data["detector"].str.replace("RWS01_", "").str.replace("MONICA_", "").str.replace("MONIBAS_", "")
data.head()

## 2.1 Do the lane detectors add up to the MONIBAS detector?

We compare the sum of the lane detectors with the MONIBAS detector at the same location,
for a few hours on a normal Tuesday (8 September).

In [ ]:
day = data[data["date"] == "2026-09-08"]
table = day.pivot_table(index="hour", columns="short", values="intensity")

def compare_lanes(lanes, monibas, hours=[3, 7, 8, 12, 17]):
    """Show the lane detectors, their sum and the MONIBAS detector side by side."""
    t = table[lanes].copy()
    t["sum of lanes"] = t[lanes].sum(axis=1)
    t["MONIBAS"] = table[monibas]
    return t.loc[hours]

In [ ]:
# Re 90.5 (eastbound, after N345): 5 lanes and one MONIBAS detector
compare_lanes(["00D0010E243CD0070007", "00D0010E243CD007000B", "00D0010E243CD007000F",
               "00D0010E243CD0070013", "00D0010E243CD0070017"], "0011hrr0905ra")

At Re 90.5 the sum of the 5 lanes is (almost) exactly the MONIBAS value: **they measure the same cars**.

In [ ]:
# Li 90.6 (westbound, before N345): lanes 0005, 0009, 000D and two extra lanes 0085, 0089
compare_lanes(["00D0010E2432D0050005", "00D0010E2432D0050009", "00D0010E2432D005000D"], "0011hrl0905ra")

In [ ]:
# The extra lanes 0085 and 0089 are NOT included in the MONIBAS value
table[["00D0010E2432D0050085", "00D0010E2432D0050089"]].loc[[3, 7, 8, 12, 17]]

At Li 90.6 the MONIBAS value equals lanes 0005 + 0009 + 000D. Lanes 0085 and 0089 carry a lot of
traffic but are not part of it, so they probably belong to another carriageway.
We leave them out.

In [ ]:
# Re 86.5 (eastbound, before Beekbergen): 5 lanes, compared with MONIBAS 0011hrr0851ra_1 (at km 85.1)
compare_lanes(["00D0010D840AD0070007", "00D0010D840AD007000B", "00D0010D840AD007000F",
               "00D0010D840AD0070087", "00D0010D840AD007008B"], "0011hrr0851ra_1")

At Re 86.5 the sum of all 5 lanes is within about 2% of the MONIBAS detector 1.4 km earlier,
so they measure the same traffic. We use the MONIBAS detector.

At **Re 43.6** and **Re 46.4** there is no MONIBAS detector. There we add up the two main lanes
(0007 and 000B). The other IDs at these locations (0087/008B at 43.6, 0187 at 46.4) are extra lanes
that we leave out. *To do: check on the Dexter map which lanes these are.*

## 2.2 Our choice per location

| Location | Direction | Detector(s) used |
|---|---|---|
| Re 43.6 | eastbound | lanes 0007 + 000B |
| Re 46.4 | eastbound | lanes 0007 + 000B |
| Re 86.5 | eastbound | MONIBAS 0011hrr0851ra_1 |
| Re 90.5 | eastbound | MONIBAS 0011hrr0905ra |
| Re 104.4 | eastbound | MONIBAS 0011hrr1044ra |
| Re 107.7 | eastbound | MONIBAS 0011hrr1076ra |
| Li 107.6 | westbound | MONIBAS 0011hrl1076ra |
| Li 98.7 | westbound | MONIBAS 0011hrl0987ra |
| Li 90.6 | westbound | MONIBAS 0011hrl0905ra |
| Li 85.6 | westbound | MONIBAS 0011hrl0856ra |
| Li 46.0 | westbound | MONIBAS 0011hrl0460ra |
| Li 43.4 | westbound | MONIBAS 0011hrl0434ra |

In [ ]:
locations = {
    "Re 43.6":  ["00D00106CC55D0070007", "00D00106CC55D007000B"],
    "Re 46.4":  ["10D00107400FD0070007", "10D00107400FD007000B"],
    "Re 86.5":  ["0011hrr0851ra_1"],
    "Re 90.5":  ["0011hrr0905ra"],
    "Re 104.4": ["0011hrr1044ra"],
    "Re 107.7": ["0011hrr1076ra"],
    "Li 107.6": ["0011hrl1076ra"],
    "Li 98.7":  ["0011hrl0987ra"],
    "Li 90.6":  ["0011hrl0905ra"],
    "Li 85.6":  ["0011hrl0856ra"],
    "Li 46.0":  ["0011hrl0460ra"],
    "Li 43.4":  ["0011hrl0434ra"],
}

## 2.3 Combine the detectors per location

- **Intensity**: we add up the detectors. If one of the lanes is missing for an hour,
  we set that hour to missing (otherwise the total would look like a drop in traffic).
- **Speed**: we take the average speed of the lanes, weighted by the number of cars per lane.
  A lane with more cars counts more. Speed 0 means "no cars measured" (see step 1), so we ignore it.

In [ ]:
data.loc[data["speed"] == 0, "speed"] = None   # speed 0 = no data

rows = []
for location, detectors in locations.items():
    sub = data[data["short"].isin(detectors)]
    for (date, hour), group in sub.groupby(["date", "hour"]):
        if len(group) < len(detectors):
            intensity, speed = None, None          # a lane is missing this hour
        else:
            intensity = group["intensity"].sum()
            valid = group.dropna(subset=["speed"])
            if valid["intensity"].sum() > 0:
                speed = (valid["speed"] * valid["intensity"]).sum() / valid["intensity"].sum()
            else:
                speed = None
        rows.append([location, date, hour, intensity, speed])

locations_data = pd.DataFrame(rows, columns=["location", "date", "hour", "intensity", "speed"])
locations_data["direction"] = locations_data["location"].str[:2].map({"Re": "eastbound", "Li": "westbound"})
locations_data.head()

## 2.4 Check

In [ ]:
# 12 locations x 7 days x 24 hours = 2016 rows (a few hours can be missing)
print(len(locations_data))
locations_data.groupby("location").size()


In [ ]:
# Quick look: total cars per location per day
locations_data.pivot_table(index="location", columns="date", values="intensity", aggfunc="sum").round(0)

In the table above **Li 85.6** has exactly 57,600 vehicles on four days. That is suspicious:
it would mean exactly 2,400 vehicles in every hour of the day. Let's look at the hourly values.

In [ ]:
locations_data[locations_data["location"] == "Li 85.6"].pivot(index="hour", columns="date", values="intensity")

In [ ]:
# Check all locations: how often does the most common intensity value occur?
# Real traffic counts almost never give exactly the same number many times.
for location, group in locations_data.groupby("location"):
    most_common = group["intensity"].value_counts().iloc[0]
    print(location, "- most common value occurs", most_common, "times")

Li 85.6 shows 2,400 vehicles/hour and 106 km/h in almost every hour, also at night.
This is not a real measurement but a fixed replacement value, probably because the detector
was broken. **We remove Li 85.6 from the analysis.** All other locations look normal.

*To do: look in Dexter for another westbound detector near junction Beekbergen (km 84-87)
and download it for the same 7 Tuesdays.*

In [ ]:
locations_data = locations_data[locations_data["location"] != "Li 85.6"]
print(locations_data["location"].nunique(), "locations left")

**Note on missing hours:** Re 43.6 has 164 instead of 168 hours, because for a few hours
both lanes were missing in the Dexter export (see step 1). These hours are simply not in the table.

## 2.5 Save the result

In [ ]:
locations_data.to_csv("../processed_data/A1_locations_hourly.csv", index=False)
print("Saved", len(locations_data), "rows")